In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("Student_Performance.csv")

In [3]:
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [4]:
df.isnull().sum()

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

In [7]:
df.duplicated().sum()

127

In [8]:
df.shape

(10000, 6)

In [9]:
def segment_production_runs(data, sample_size, n=3):
    production_data = dict()
    for i in range(n):
        production_data[i+1] = data.sample(sample_size, random_state=42)
        remove_data = production_data[i+1].index.tolist()
        data = data.drop(remove_data,axis=0)
    return data, production_data

In [10]:
data, test = segment_production_runs(df, sample_size=1000, n=1)

In [11]:
data.shape, test[1].shape

((9000, 6), (1000, 6))

In [12]:
data.to_csv("data.csv")

In [16]:
test[1].to_csv("test.csv")

In [17]:
data.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
4,7,75,No,8,5,66.0
5,3,78,No,9,6,61.0
6,7,73,Yes,5,6,63.0


In [18]:
test_df = pd.read_csv("test.csv")

In [19]:
test_df.isnull().sum()

Unnamed: 0                           0
Hours Studied                       15
Previous Scores                     15
Extracurricular Activities           0
Sleep Hours                         29
Sample Question Papers Practiced    29
Performance Index                    0
dtype: int64

In [21]:
test_df.duplicated().sum()

3

In [22]:
data.duplicated().sum()

103

In [23]:
data.drop_duplicates(inplace=True)

In [24]:
data.duplicated().sum()

0

In [27]:
for i in test_df:
    print(i, test_df[i].dtype)

Unnamed: 0 int64
Hours Studied object
Previous Scores float64
Extracurricular Activities object
Sleep Hours float64
Sample Question Papers Practiced float64
Performance Index int64


In [60]:
for i in data:
    print(i, data[i].dtype)

Hours Studied int64
Previous Scores int64
Extracurricular Activities object
Sleep Hours int64
Sample Question Papers Practiced int64
Performance Index float64


In [28]:
dtypes_issue = dict()

for i in data:
    [test_df[i].iloc[j].dtype]

Hours Studied
Previous Scores
Extracurricular Activities
Sleep Hours
Sample Question Papers Practiced
Performance Index


In [30]:
np.array([True, False, True]).sum()

2

In [47]:
check = []
for i in test_df['Extracurricular Activities']:
    check.append(type(test_df['Extracurricular Activities'].iloc[0]) != type(data['Extracurricular Activities'].iloc[0]))

In [48]:
print(check[:5])

[False, False, False, False, False]


In [49]:
np.array(check).sum(), len(check)

(0, 1000)

In [45]:
type(data['Extracurricular Activities'].iloc[0])

str

In [46]:
type(test_df['Sleep Hours'].iloc[0])

numpy.float64

In [66]:
mismatch, actual_type, got = [], [], []
for i in data:
    
    if type(data[i].iloc[0]) != type(test_df[i].iloc[0]):
        mismatch.append(i)
        actual_type.append(type(data[i].iloc[0]))
        got.append(type(test_df[i].iloc[0]))

pd.DataFrame({'Mismatch Col':mismatch,
             "Baseline Type":actual_type,
             "Production Type":got}).head()

Hours Studied
Previous Scores
Sleep Hours
Sample Question Papers Practiced
Performance Index


,Mismatch,Baseline Type,Production Type
0,Hours Studied,<class 'numpy.int64'>,<class 'str'>
1,Previous Scores,<class 'numpy.int64'>,<class 'numpy.float64'>
2,Sleep Hours,<class 'numpy.int64'>,<class 'numpy.float64'>
3,Sample Question Papers Practiced,<class 'numpy.int64'>,<class 'numpy.float64'>
4,Performance Index,<class 'numpy.float64'>,<class 'numpy.int64'>


In [64]:
for i in data:
    print(type(data[i].iloc[0]))

<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'str'>
<class 'numpy.int64'>
<class 'numpy.int64'>
<class 'numpy.float64'>


In [68]:
np.where(test_df['Extracurricular Activities'].unique() not in data['Extracurricular Activities'].unique())

<ipython-input-68-37b94c6a0c12>:1: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  np.where(test_df['Extracurricular Activities'].unique() not in data['Extracurricular Activities'].unique())


(array([0], dtype=int64),)

In [75]:
def replace_non_dtypes(value,target_type):
    if isinstance(value, target_type):
        return value
    else:
        return np.nan
    
def data_validity(df):
    missing_vals, wrong_dtypes = dict(), dict()
    for i in df:
        missing_vals[i] = {"count" : df[i].isnull().sum(), "index": df[df[i].isnull()].index}
    spare_df = df.copy()
    
    for i in data:
        spare_df[i] = df[i].apply(replace_non_dtypes,args=(type(data[i].iloc[0])),)
#     spare_df['Weight'] = df['Weight'].apply(replace_non_dtypes,args=(str,))
#     spare_df['BMI'] = df['BMI'].apply(replace_non_dtypes,args=(int,))
    for i in spare_df:
        indices = set(spare_df[spare_df[i].isnull()].index) - set(missing_vals[i]["index"])
        wrong_dtypes[i] = {"count": len(indices), "index": indices}
      
       
    missing_count, wrong_dtypes_count, = 0,0
    for i in df:
        missing_count += missing_vals[i]["count"]
        wrong_dtypes_count += wrong_dtypes[i]["count"]
    return missing_vals, wrong_dtypes, missing_count, wrong_dtypes_count, round((missing_count + wrong_dtypes_count) / df.size, 2)


In [76]:
miss, wrong_dtypes, miss_c, wrong_dtype_c, score = data_validity(test_df)

TypeError: replace_non_dtypes() argument after * must be an iterable, not type

In [183]:
def validity_check(baseline_data, production_data):
    
    baseline_data = baseline_data.loc[:, ~baseline_data.columns.str.contains('^Unnamed')]
    common_columns = baseline_data.columns.intersection(production_data.columns)
    production_data = production_data[common_columns]
    
    miss_dict = {i:[] for i in baseline_data.columns}
    cat_has_int = {i:[] for i in baseline_data.select_dtypes(include='object').columns}
    int_has_cat = {i:[] for i in baseline_data.select_dtypes(exclude='object').columns}
    indices = []
    outliers_index = {i:[] for i in baseline_data.columns}
    print(miss_dict, cat_has_int, int_has_cat)
    
    if list(baseline_data.columns) == list(production_data.columns):
        for column in baseline_data.columns:
            
            if production_data[column].isnull().any():
                missing_indices = production_data[production_data[column].isnull()].index.tolist()
                
                miss_dict[column] = missing_indices
            
            if production_data[column].dtype == 'object' and baseline_data[column].dtype == 'object':
                unique_bs = set(baseline_data[column].unique())
                unique_pd = set(production_data[column].unique())
                extras = unique_bs.symmetric_difference(unique_pd)
                
                cat_has_int_id = []
                for i in extras:
                    index_list = production_data[production_data[column]  ==  i].index.tolist()
                    cat_has_int_id.extend(index_list)
                cat_has_int[column] = cat_has_int_id
            
            if baseline_data[column].dtype != production_data[column].dtype:
                int_has_cat_id = []
                for index, value in production_data[column].items():
                    
                    try:
                        float(value)
                    except ValueError:
                        int_has_cat_id.append(index)
                int_has_cat[column] = int_has_cat_id
                        
    for k in list(miss_dict.keys()):
        indices.extend(miss_dict[k])
    
    for k in list(cat_has_int.keys()):
        indices.extend(cat_has_int[k])
    
    for k in list(int_has_cat.keys()):
        indices.extend(int_has_cat[k])
    
    num_invalid = len(set(indices))
    total_rows = len(production_data)
    score = np.round((total_rows - num_invalid) / total_rows * 100, 2)
    
    if num_invalid != 0:
        clean_df = production_data.drop(indices, axis=0)
        print([(i, clean_df[i].dtype) for i in clean_df])
        outliers_index = outliers_for_num_data(clean_df.select_dtypes(exclude='object'))
    
    return int_has_cat, cat_has_int, miss_dict, score, indices, outliers_index, num_invalid

In [187]:
int_has_cat, cat_has_int, miss_dict, score, indices, outliers_index, num_invalid = validity_check(data, test_df)

{'Hours Studied': [], 'Previous Scores': [], 'Extracurricular Activities': [], 'Sleep Hours': [], 'Sample Question Papers Practiced': [], 'Performance Index': []} {'Extracurricular Activities': []} {'Hours Studied': [], 'Previous Scores': [], 'Sleep Hours': [], 'Sample Question Papers Practiced': [], 'Performance Index': []}
[('Hours Studied', dtype('O')), ('Previous Scores', dtype('float64')), ('Extracurricular Activities', dtype('O')), ('Sleep Hours', dtype('float64')), ('Sample Question Papers Practiced', dtype('float64')), ('Performance Index', dtype('int64'))]


In [188]:
miss_dict, cat_has_int, int_has_cat

({'Hours Studied': [815,
   816,
   817,
   818,
   819,
   820,
   821,
   822,
   823,
   824,
   825,
   826,
   827,
   828,
   829],
  'Previous Scores': [815,
   816,
   817,
   818,
   819,
   820,
   821,
   822,
   823,
   824,
   825,
   826,
   827,
   828,
   829],
  'Extracurricular Activities': [],
  'Sleep Hours': [722,
   723,
   724,
   825,
   826,
   827,
   828,
   829,
   830,
   831,
   832,
   833,
   834,
   835,
   836,
   837,
   838,
   839,
   840,
   841,
   842,
   843,
   844,
   845,
   846,
   847,
   848,
   849,
   850],
  'Sample Question Papers Practiced': [722,
   723,
   724,
   825,
   826,
   827,
   828,
   829,
   830,
   831,
   832,
   833,
   834,
   835,
   836,
   837,
   838,
   839,
   840,
   841,
   842,
   843,
   844,
   845,
   846,
   847,
   848,
   849,
   850],
  'Performance Index': []},
 {'Extracurricular Activities': [133, 49, 50, 51, 52, 53, 54, 55, 56, 57]},
 {'Hours Studied': [154,
   155,
   156,
   157,
   158,
   159,


In [175]:
int_has_cat, cat_has_int, miss_dict, score, indices, outliers_index

({}, {'Extracurricular Activities': []}, {}, 100.0, [], {})

In [167]:
data.shape

(8897, 6)

In [147]:
len(indices), score, len(test_df)

(110, 93.9, 1000)

In [81]:
values

{'column': ['Hours Studied',
  'Hours Studied',
  'Previous Scores',
  'Previous Scores',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Sleep Hours',
  'Sleep Hours',
  'Sample Question Papers Practiced',
  'Sample Question Papers Practiced',
  'Performance Index'],
 'Baseline Datatype': [dtype('int64'),
  dtype('int64'),
  dtype('int64'),
  dtype('int64'),
  dtype('O'),
  dtype('O'),
  dtype('O'),
 

In [83]:
total_rows, invalid_rows, num_invalid, score

(1000, 939, 61, 93.9)

In [85]:
cat_err

{'column': ['Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities',
  'Extracurricular Activities'],
 'indices': [49, 50, 51, 52, 53, 54, 55, 56, 57, 133]}

In [107]:
str(1)

'1'

In [106]:
cat_has_int

{'Extracurricular Activities': [133, 49, 50, 51, 52, 53, 54, 55, 56, 57]}

In [86]:
test_df.iloc[cat_err['indices']]

,Unnamed: 0,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
49,2304,5,57.0,1,5.0,8.0,39
50,3723,1,96.0,1,8.0,5.0,72
51,8284,3,77.0,1,8.0,4.0,57
52,4993,3,91.0,1,7.0,9.0,74
53,8127,8,89.0,1,5.0,8.0,88
54,3032,9,77.0,1,5.0,7.0,70
55,7938,8,95.0,1,6.0,6.0,89
56,3039,9,77.0,1,7.0,3.0,74
57,9655,8,93.0,1,8.0,3.0,86
133,5794,5,81.0,157,5.0,7.0,67


In [90]:
miss_dict

{'Hours Studied': {'count': 15,
  'indices': [815,
   816,
   817,
   818,
   819,
   820,
   821,
   822,
   823,
   824,
   825,
   826,
   827,
   828,
   829]},
 'Previous Scores': {'count': 15,
  'indices': [815,
   816,
   817,
   818,
   819,
   820,
   821,
   822,
   823,
   824,
   825,
   826,
   827,
   828,
   829]},
 'Sleep Hours': {'count': 29,
  'indices': [722,
   723,
   724,
   825,
   826,
   827,
   828,
   829,
   830,
   831,
   832,
   833,
   834,
   835,
   836,
   837,
   838,
   839,
   840,
   841,
   842,
   843,
   844,
   845,
   846,
   847,
   848,
   849,
   850]},
 'Sample Question Papers Practiced': {'count': 29,
  'indices': [722,
   723,
   724,
   825,
   826,
   827,
   828,
   829,
   830,
   831,
   832,
   833,
   834,
   835,
   836,
   837,
   838,
   839,
   840,
   841,
   842,
   843,
   844,
   845,
   846,
   847,
   848,
   849,
   850]}}

In [91]:
for i in test_df:
    print(i, test_df[i].isnull().sum())

Unnamed: 0 0
Hours Studied 15
Previous Scores 15
Extracurricular Activities 0
Sleep Hours 29
Sample Question Papers Practiced 29
Performance Index 0


In [95]:
{1,2,3,4}.symmetric_difference({3,4,5,6})

{1, 2, 5, 6}

In [96]:
{3,4,5,6} - {1,2,3,4}

{5, 6}

In [100]:
for i in test_df['Extracurricular Activities'].unique():
    print(test_df[test_df['Extracurricular Activities'] == i].index.tolist())

[0, 4, 5, 7, 8, 12, 15, 16, 18, 20, 22, 23, 25, 26, 33, 35, 38, 43, 45, 48, 58, 59, 60, 61, 62, 65, 70, 71, 72, 74, 76, 78, 79, 81, 83, 84, 86, 87, 88, 89, 92, 93, 95, 97, 98, 101, 105, 107, 108, 110, 114, 115, 119, 120, 121, 124, 128, 132, 134, 136, 137, 139, 141, 142, 145, 148, 149, 150, 151, 153, 154, 155, 158, 159, 163, 164, 167, 169, 171, 172, 174, 175, 176, 179, 181, 182, 184, 187, 189, 191, 192, 197, 198, 203, 208, 209, 210, 211, 212, 214, 215, 216, 218, 221, 222, 223, 224, 226, 229, 234, 235, 237, 238, 241, 242, 245, 246, 248, 249, 253, 254, 255, 256, 257, 258, 259, 260, 261, 264, 265, 266, 268, 270, 272, 273, 274, 278, 283, 286, 288, 289, 291, 292, 294, 296, 298, 301, 302, 305, 306, 307, 311, 312, 314, 315, 317, 319, 321, 324, 330, 332, 334, 342, 344, 345, 346, 347, 348, 350, 351, 353, 354, 357, 360, 361, 362, 363, 364, 366, 367, 368, 369, 372, 377, 378, 382, 383, 384, 387, 388, 390, 391, 393, 394, 396, 397, 398, 399, 400, 401, 402, 404, 406, 407, 408, 410, 412, 421, 422, 423,

In [108]:
test_df['Hours Studied'].unique()

array(['5', '2', '7', '6', '9', '4', '8', '1', '3', 'A', nan],
      dtype=object)

In [123]:
test_df.iloc[int_has_cat['Hours Studied']]

,Unnamed: 0,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
154,5269,A,92.0,No,9.0,5.0,70
155,6897,A,42.0,No,8.0,8.0,20
156,5309,A,54.0,Yes,4.0,3.0,35
157,31,A,44.0,Yes,9.0,1.0,36
158,5674,A,72.0,No,7.0,5.0,50
159,4386,A,40.0,No,4.0,6.0,19
160,29,A,90.0,Yes,4.0,3.0,74
161,4202,A,54.0,Yes,4.0,5.0,30
162,7507,A,92.0,Yes,4.0,3.0,67
163,735,A,41.0,No,5.0,9.0,34


In [126]:
test_df.iloc[int_has_cat['Hours Studied']]

,Unnamed: 0,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
154,5269,A,92.0,No,9.0,5.0,70
155,6897,A,42.0,No,8.0,8.0,20
156,5309,A,54.0,Yes,4.0,3.0,35
157,31,A,44.0,Yes,9.0,1.0,36
158,5674,A,72.0,No,7.0,5.0,50
159,4386,A,40.0,No,4.0,6.0,19
160,29,A,90.0,Yes,4.0,3.0,74
161,4202,A,54.0,Yes,4.0,5.0,30
162,7507,A,92.0,Yes,4.0,3.0,67
163,735,A,41.0,No,5.0,9.0,34


In [127]:
float(np.nan)

nan

In [133]:
test_df.drop("Unnamed: 0", axis=1, inplace=True)

In [140]:
indices = []
for k in list(miss_dict.keys()):
    indices.extend(miss_dict[k]['indices'])

In [142]:
len(indices), len(set(indices))

(88, 39)

In [154]:
def outliers_for_num_data(data):
    
    outliers_indices = {}
    
    for i in data.columns:
        
        Q1 = data[i].quantile(0.25)
        Q3 = data[i].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = data[(data[i] < lower_bound) | (data[i] > upper_bound)]
        outliers_indices[i] = outliers.index.tolist()
    
    return outliers_indices

In [190]:
test_df.drop(indices, axis=0).shape, test_df.shape

((939, 6), (1000, 6))